# 1st Assignment

Filo logico:

Per le due condizioni distinte:

* raccogliere per diametro (di tornitura) -> {potenza} C&NC
* [Ptot] -> [Pc] -> [Fc] -> [Kc] -> KC_avg

In [1]:
# Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
import numpy as np

In [2]:
# Tool parameters

ap = 1  #(mm)
f = 0.2 #(mm)
Vc = 150    #(m/min)
N = 1200    #(rpm)

## Loading data

In [3]:
# File data CSV -> C:= cutting; NC := NON-cutting
filedataC150 = 'ANNACHPTVC150.csv'
filedataNC150 = 'ANNACHPTVC150VIDE.csv'
filedataC1200 = 'ANNACHPT1200RPM.csv'
filedataNC1200 = 'ANNACHPT1200RPMVIDE.csv'

In [4]:
# Reading csv
dfC150 = pd.read_csv(filedataC150, skiprows=10)
dfNC150 = pd.read_csv(filedataNC150, skiprows=10)
dfC1200 = pd.read_csv(filedataC1200, skiprows=10)
dfNC1200 = pd.read_csv(filedataNC1200, skiprows=10)

In [5]:
# Naming columns
dfC150.columns = [
    'time',
    'power',
    'prog_pos_z',   # Programmed position
    'prog_pos_x',
    'meas_pos_z',   # Actual value of position 
    'meas_pos_x',
    'spindle1'  # Spindle speed. Only the first data does not have the spindle 2 data
]

cols = ['time', 'power', 'prog_pos_z', 'prog_pos_x',
        'meas_pos_z', 'meas_pos_x', 'spindle1', 'spindle2']
for df in [dfNC150, dfC1200, dfNC1200]:
    df.columns = cols

In [6]:
# Defining indecis of columns
cols_to_convert = dfC150.columns.difference(['time'])

# Conversion to numeric type
for df in [dfC150, dfNC150, dfC1200, dfNC1200]:
    for col in cols_to_convert:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Suppressing Nan columns
for df in [dfC150, dfNC150, dfC1200, dfNC1200]:
    df.dropna(inplace=True)

## Filtering data per programmed cutting X position

In [7]:
# ==== Energy calculation ====

# Calculating dt
for df in [dfC150, dfNC150, dfC1200, dfNC1200]:
    df['delta_t'] = df['time'].diff()
    df.loc[df.index[0], 'delta_t'] = 0

# Instantaneous energy (W x s / 1000 = kJ)
for df in [dfC150, dfNC150, dfC1200, dfNC1200]:
    df['energy_kJ'] = df['power'] * df['delta_t'] / 1000

In [ ]:
# Vector in order to do for-loops
dfs = {'C150': dfC150, 'NC150': dfNC150, 'C1200': dfC1200, 'NC1200': dfNC1200}

# Filter data
for key, df in dfs.items():
    dfs[key] = df[(df['prog_pos_x'] % 1 == 0) &
                  (df['prog_pos_x'] >= 24) &
                  (df['prog_pos_x'] <= 46)]
    
dfC150 = dfs['C150']
dfNC150 = dfs['NC150']
dfC1200 = dfs['C1200']
dfNC1200 = dfs['NC1200']

## Cutting power

$P_c[i] = P[i]|_{_{C-}} - P[i]|_{_{NC-}}$

In [11]:
df_150 = pd.DataFrame({
    'Pc': dfC150['power'] - dfNC1200['power']
})